# Data Preprocessing
## Text Cleaning & Train/Val/Test Split

Chuẩn bị dữ liệu cho training: làm sạch text, loại bỏ duplicates, chia dữ liệu.


In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from src.utils import clean_text, set_seed
from src.config import RAW_DATA_FILE, PROCESSED_DIR, SEED
import os
import json
from datetime import datetime

set_seed(SEED)
print('Libraries loaded')


Libraries loaded


## 1. Load Raw Data


In [2]:
df = pd.read_csv(RAW_DATA_FILE)
print(f'Raw data: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head()


Raw data: (136122, 2)
Columns: ['content', 'label']


,content,label
0,Foreign Democrat final. more tax development b...,0
1,To offer down resource great point. probably g...,1
2,Himself church myself carry. them identify for...,1
3,You unit its should. phone which item yard Rep...,1
4,Billion believe employee summer how. wonder my...,1


## 2. Data Cleaning


In [3]:
text_col = 'content'  # Adjust if column name differs
label_col = 'label'

print(f'Before cleaning: {len(df):,} records')

# Drop missing text
df = df.dropna(subset=[text_col])
print(f'After dropna:    {len(df):,} records')

# Drop duplicates
df = df.drop_duplicates(subset=[text_col])
print(f'After dedup:     {len(df):,} records')

# Store original length
df['original_length'] = df[text_col].astype(str).apply(len)

# Apply text cleaning
df['text_cleaned'] = df[text_col].astype(str).apply(clean_text)
df['cleaned_length'] = df['text_cleaned'].apply(len)

# Remove empty texts after cleaning
df = df[df['cleaned_length'] > 10]
print(f'After cleaning:  {len(df):,} records')

print(f'\nAvg original length: {df["original_length"].mean():.0f} chars')
print(f'Avg cleaned length:  {df["cleaned_length"].mean():.0f} chars')
print(f'Length reduction:     {(1 - df["cleaned_length"].mean()/df["original_length"].mean())*100:.1f}%')


Before cleaning: 136,122 records
After dropna:    136,121 records
After dedup:     136,121 records
After cleaning:  136,032 records

Avg original length: 2596 chars
Avg cleaned length:  2560 chars
Length reduction:     1.4%


## 3. Prepare Final DataFrame


In [4]:
# Use cleaned text
df_final = df[[label_col, 'text_cleaned']].copy()
df_final.columns = ['label', 'text']

# Encode labels if needed (ensure 0=real, 1=fake)
if df_final['label'].dtype == 'object':
    from src.config import LABEL_MAP
    df_final['label'] = df_final['label'].map(LABEL_MAP)

print(f'Final dataset: {len(df_final):,} records')
print(f'\nLabel distribution:')
print(df_final['label'].value_counts())


Final dataset: 136,032 records

Label distribution:
label
0    72244
1    63788
Name: count, dtype: int64


## 4. Train/Val/Test Split (70/15/15)


In [5]:
# Stratified split
train_df, temp_df = train_test_split(
    df_final, test_size=0.30, random_state=SEED,
    stratify=df_final['label']
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df['label']
)

print(f'Train: {len(train_df):,} ({len(train_df)/len(df_final)*100:.1f}%)')
print(f'Val:   {len(val_df):,} ({len(val_df)/len(df_final)*100:.1f}%)')
print(f'Test:  {len(test_df):,} ({len(test_df)/len(df_final)*100:.1f}%)')

# Verify stratification
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    fake_pct = split['label'].mean() * 100
    print(f'  {name} fake%: {fake_pct:.1f}%')


Train: 95,222 (70.0%)
Val:   20,405 (15.0%)
Test:  20,405 (15.0%)
  Train fake%: 46.9%
  Val fake%: 46.9%
  Test fake%: 46.9%


## 5. Save Processed Data


In [6]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

train_df.to_csv(os.path.join(PROCESSED_DIR, 'train.csv'), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, 'val.csv'), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, 'test.csv'), index=False)

# Save preprocessing report
report = {
    'preprocessing_date': datetime.now().isoformat(),
    'total_records': len(df_final),
    'splits': {
        'train': {'count': len(train_df), 'percentage': len(train_df)/len(df_final)*100,
                  'fake_percentage': train_df['label'].mean()*100},
        'val': {'count': len(val_df), 'percentage': len(val_df)/len(df_final)*100,
                'fake_percentage': val_df['label'].mean()*100},
        'test': {'count': len(test_df), 'percentage': len(test_df)/len(df_final)*100,
                 'fake_percentage': test_df['label'].mean()*100}
    },
    'text_statistics': {
        'avg_original_length': float(df['original_length'].mean()),
        'avg_cleaned_length': float(df['cleaned_length'].mean()),
    },
    'preprocessing_config': {
        'remove_urls': True, 'remove_mentions': True,
        'remove_hashtags': True, 'random_seed': SEED
    }
}

with open(os.path.join(PROCESSED_DIR, 'preprocessing_report.json'), 'w') as f:
    json.dump(report, f, indent=2)

print('All files saved to:', PROCESSED_DIR)
print('  - train.csv')
print('  - val.csv')
print('  - test.csv')
print('  - preprocessing_report.json')


All files saved to: d:\Fake_news_RoBERTa\data\processed
  - train.csv
  - val.csv
  - test.csv
  - preprocessing_report.json


## 6. Verification


In [7]:
# Verify saved files
for name in ['train', 'val', 'test']:
    path = os.path.join(PROCESSED_DIR, f'{name}.csv')
    check = pd.read_csv(path)
    print(f'{name}.csv: {check.shape} | null text: {check["text"].isnull().sum()} | labels: {check["label"].value_counts().to_dict()}')

print('\nPreprocessing complete!')


train.csv: (95222, 2) | null text: 0 | labels: {0: 50571, 1: 44651}
val.csv: (20405, 2) | null text: 0 | labels: {0: 10836, 1: 9569}
test.csv: (20405, 2) | null text: 0 | labels: {0: 10837, 1: 9568}

Preprocessing complete!
